Este notebook contém a solução do OHS utilizando Runge-Kutta. 
\
Além disso, gera uma base de dados com as posições e velocidades do OHS no tempo, obtendo assim o espaço de fase do sistema.

In [42]:
import numpy as np
import pandas as pd
import torch
import locale
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import random
from datetime import datetime

# define o formato padrão brasileiro numérico dos valores da base
try:
    locale.setlocale(locale.LC_NUMERIC, 'pt_BR.UTF-8')
except:
    try:
        locale.setlocale(locale.LC_NUMERIC, 'Portuguese_Brazil.1252')
    except:
        print("Formato pt-br não disponível, usando formato padrão")
        locale.setlocale(locale.LC_NUMERIC, 'C')

# paleta de cores dos gráficos
CORES_PALETA = [
    '#FF1493',  # DeepPink
    '#00FF00',  # Lime
    '#FF4500',  # OrangeRed
    '#00BFFF',  # DeepSkyBlue
    '#FFD700',  # Gold
    '#8B00FF',  # DarkViolet
    '#FF6347',  # Tomato
    '#00FA9A',  # MediumSpringGreen
    '#DC143C',  # Crimson
    '#1E90FF',  # DodgerBlue
    '#FF8C00',  # DarkOrange
    '#32CD32',  # LimeGreen
    '#FF00FF',  # Magenta
    '#00CED1',  # DarkTurquoise
    '#FF69B4',  # HotPink
    '#7FFF00',  # Chartreuse
    '#8A2BE2',  # BlueViolet
    '#00FF7F',  # SpringGreen
    '#FF2400',  # Scarlet
    '#0000CD',  # MediumBlue
]

class OsciladorHarmonicoPyTorch:
    """
    Classe para resolver a equação do oscilador harmônico simples usando PyTorch e Runge-Kutta
    """
    
    def __init__(self, frequencias_angulares, amortecimento=0.0, device='cpu'):
        """
        Parâmetros do oscilador para múltiplos sistemas simultâneos
        
        Args:
            frequencias_angulares (list ou tensor): frequências angulares de cada sistema (rad/s)
            amortecimento (float): coeficiente de amortecimento (kg/s)
            device (str): dispositivo para computação ('cpu' ou 'cuda')
        """
        # converte para tensor (n_sistemas,)
        if isinstance(frequencias_angulares, list):
            self.frequencias_angulares = torch.tensor(frequencias_angulares, dtype=torch.float32, device=device)
        else:
            self.frequencias_angulares = frequencias_angulares.clone().detach().to(device)
        
        self.device = device
        self.amortecimento = amortecimento
        self.n_sistemas = len(self.frequencias_angulares)
        self.b = torch.tensor(amortecimento, dtype=torch.float32, device=device).expand(self.n_sistemas)
        self.frequencias_lineares = self.frequencias_angulares / (2 * np.pi)
        self.periodos = 1.0 / self.frequencias_lineares
        
    def equacoes_movimento(self, estados):
        """
        Define as equações do movimento para múltiplas condições iniciais e múltiplos sistemas
        
        estados: tensor de forma (n_condicoes, n_sistemas, 2)
        retorna: tensor de forma (n_condicoes, n_sistemas, 2)
        """
        x = estados[:, :, 0]
        v = estados[:, :, 1]
        
        dxdt = v
        
        # rearanja omega^2 para que tenha as dimensões condizentes com o número de linhas e sistemas (N, n_sistemas)
        omega2 = self.frequencias_angulares ** 2
        omega2_expanded = omega2.unsqueeze(0).expand(x.shape[0], -1)
        b_expanded = self.b.unsqueeze(0).expand(x.shape[0], -1)
        
        dvdt = -omega2_expanded * x - b_expanded * v
        
        return torch.stack([dxdt, dvdt], dim=2)
    
    def runge_kutta_4(self, estados, dt):
        """
        Método Runge-Kutta de 4ª ordem para múltiplas condições iniciais e múltiplos sistemas
        """
        k1 = self.equacoes_movimento(estados)
        k2 = self.equacoes_movimento(estados + 0.5 * dt * k1)
        k3 = self.equacoes_movimento(estados + 0.5 * dt * k2)
        k4 = self.equacoes_movimento(estados + dt * k3)
        
        estados_novos = estados + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)
        
        return estados_novos
        
    def resolve_multi_condicoes_sistemas(self, condicoes_iniciais, t_final, dt):
        """
        Resolve a EDO para múltiplas condições iniciais e múltiplos sistemas simultaneamente
        
        condicoes_iniciais: tensor de forma (n_condicoes, 2) com [x0, v0]
        t_final: tempo final
        dt: passo temporal
        """
        n_condicoes = condicoes_iniciais.shape[0]
        n_passos = int(t_final / dt) + 1
        
        tempos = torch.linspace(0, t_final, n_passos, device=self.device)
        
        # inicializa tensores 3D: (n_passos, n_condicoes, n_sistemas)
        posicoes = torch.zeros((n_passos, n_condicoes, self.n_sistemas), device=self.device)
        velocidades = torch.zeros((n_passos, n_condicoes, self.n_sistemas), device=self.device)
        
        # rearanja condições_iniciais para que tenha as dimensões condizentes com o número de linhas e sistemas (N, n_sistemas)
        cond_iniciais_expand = condicoes_iniciais.unsqueeze(1).expand(-1, self.n_sistemas, -1)
        
        estados = cond_iniciais_expand.clone()
        posicoes[0] = estados[:, :, 0]
        velocidades[0] = estados[:, :, 1]
        
        for i in range(1, n_passos):
            estados = self.runge_kutta_4(estados, dt)
            posicoes[i] = estados[:, :, 0]
            velocidades[i] = estados[:, :, 1]
        
        # calcula energias (normalizadas por massa) para cada sistema
        omega2 = self.frequencias_angulares ** 2

        # rearanja omega2 para que tenha as dimensões condizentes com o número de linhas, passo temporal e condições iniciais (N, n_passos, n_condicoes)
        omega2_expanded = omega2.unsqueeze(0).unsqueeze(0).expand(n_passos, n_condicoes, -1)
        
        energia_cinetica = 0.5 * velocidades**2
        energia_potencial = 0.5 * omega2_expanded * posicoes**2
        energia_mecanica = energia_cinetica + energia_potencial
        
        # calcula amplitudes máximas para cada condição inicial e sistema
        amplitudes_max = torch.max(torch.abs(posicoes), dim=0)[0].cpu().numpy()
        
        return {
            'tempo': tempos.cpu().numpy(),
            'posicao': posicoes.cpu().numpy(),
            'velocidade': velocidades.cpu().numpy(),
            'energia_cinetica': energia_cinetica.cpu().numpy(),
            'energia_potencial': energia_potencial.cpu().numpy(),
            'energia_mecanica': energia_mecanica.cpu().numpy(),
            'amplitudes': amplitudes_max,
            'condicoes_iniciais': condicoes_iniciais.cpu().numpy(),
            'frequencias_angulares': self.frequencias_angulares.cpu().numpy(),
            'frequencias_lineares': self.frequencias_lineares.cpu().numpy(),
            'periodos': self.periodos.cpu().numpy(),
            'n_condicoes': n_condicoes,
            'n_sistemas': self.n_sistemas
        }
    
    def formata_numero_pt_br(self, numero):
        """
        Formata número no padrão brasileiro
        """
        if isinstance(numero, (int, float)):
            if abs(numero) < 1000:
                return f"{numero:.3f}".replace('.', ',')
            else:
                partes = f"{numero:.3f}".split('.')
                parte_inteira = '{:,.0f}'.format(int(partes[0])).replace(',', '.')
                return f"{parte_inteira},{partes[1]}"
        return str(numero)
    
    def gera_base_dados(self, condicoes_iniciais, t_final, dt, sistemas_descricao):
        """
        Gera uma base de dados completa com as soluções no formato brasileiro
        
        Args:
            condicoes_iniciais: tensor com condições iniciais (n_condicoes, 2)
            t_final: tempo final
            dt: passo temporal
            sistemas_descricao: lista com descrições dos sistemas
        """
        solucao = self.resolve_multi_condicoes_sistemas(condicoes_iniciais, t_final, dt)
        
        dados = []
        n_condicoes = solucao['n_condicoes']
        n_sistemas = solucao['n_sistemas']
        n_passos = len(solucao['tempo'])
        
        for i_sistema in range(n_sistemas):
            for i_cond in range(n_condicoes):
                for j in range(n_passos):
                    dados.append({
                        'sistema_id': i_sistema,
                        'simulacao_id': i_cond,
                        'tempo': self.formata_numero_pt_br(solucao['tempo'][j]),
                        'posicao': self.formata_numero_pt_br(solucao['posicao'][j, i_cond, i_sistema]),
                        'velocidade': self.formata_numero_pt_br(solucao['velocidade'][j, i_cond, i_sistema]),
                        'descricao_sistema': sistemas_descricao[i_sistema],
                        'frequencia_angular': self.formata_numero_pt_br(solucao['frequencias_angulares'][i_sistema]),
                        'frequencia_linear': self.formata_numero_pt_br(solucao['frequencias_lineares'][i_sistema]),
                        'periodo_s': self.formata_numero_pt_br(solucao['periodos'][i_sistema]),
                        'x0': self.formata_numero_pt_br(solucao['condicoes_iniciais'][i_cond, 0]),
                        'v0': self.formata_numero_pt_br(solucao['condicoes_iniciais'][i_cond, 1]),
                        'amplitude_max': self.formata_numero_pt_br(solucao['amplitudes'][i_cond, i_sistema]),
                        'energia_cinetica': self.formata_numero_pt_br(solucao['energia_cinetica'][j, i_cond, i_sistema]),
                        'energia_potencial': self.formata_numero_pt_br(solucao['energia_potencial'][j, i_cond, i_sistema]),
                        'energia_mecanica': self.formata_numero_pt_br(solucao['energia_mecanica'][j, i_cond, i_sistema]),
                    })
        
        return pd.DataFrame(dados), solucao

def gera_condicoes_iniciais_aleatorias(n_simulacoes, x0_limites, v0_limites, seed=None):
    """
    Gera condições iniciais bem distribuídas usando amostragem estratificada no espaço de fases
    
    Args:
        n_simulacoes: número de simulações (condições iniciais)
        x0_limites: tupla (min, max) para posição inicial
        v0_limites: tupla (min, max) para velocidade inicial
        seed: semente para reprodutibilidade
    """
    if seed is not None:
        np.random.seed(seed)
        torch.manual_seed(seed)
    
    x0_estratos = np.linspace(x0_limites[0], x0_limites[1], n_simulacoes + 1)
    v0_estratos = np.linspace(v0_limites[0], v0_limites[1], n_simulacoes + 1)
    
    x0 = []
    v0 = []
    
    for i in range(n_simulacoes):
        x0_val = np.random.uniform(x0_estratos[i], x0_estratos[i+1])
        v0_val = np.random.uniform(v0_estratos[i], v0_estratos[i+1])
        
        x0.append(x0_val)
        v0.append(v0_val)
    
    # embaralha para não ficar na ordem correlacionada
    indices = np.random.permutation(n_simulacoes)
    x0 = np.array(x0)[indices]
    v0 = np.array(v0)[indices]
    
    return torch.tensor(np.column_stack([x0, v0]), dtype=torch.float32)

def gera_frequencias_angulares_aleatorias(n_sistemas, omega_min, omega_max, seed=None):
    """
    Gera frequências angulares bem distribuídas usando amostragem estratificada
    
    Args:
        n_sistemas: número de sistemas diferentes
        omega_min: frequência angular mínima (rad/s)
        omega_max: frequência angular máxima (rad/s)
        seed: semente para reprodutibilidade
    """
    if seed is not None:
        np.random.seed(seed)
    
    estratos = np.linspace(omega_min, omega_max, n_sistemas + 1)
    
    omegas = []
    for i in range(n_sistemas):
        omega = np.random.uniform(estratos[i], estratos[i+1])
        omegas.append(omega)
    
    # embaralha para não ficar na ordem crescente
    np.random.shuffle(omegas)
    
    return omegas

def cria_grafico_3d(solucao, sistemas_descricao):
    """
    Cria visualização 3D com todas as trajetórias de todos os sistemas no espaço de fases
    """
    fig = go.Figure()
    
    n_sistemas = solucao['n_sistemas']
    n_condicoes = solucao['n_condicoes']
        
    for i_sistema in range(n_sistemas):
        cor = CORES_PALETA[i_sistema % len(CORES_PALETA)]
        freq = solucao['frequencias_lineares'][i_sistema]
        omega = solucao['frequencias_angulares'][i_sistema]
        
        for i_cond in range(n_condicoes):
            x0, v0 = solucao['condicoes_iniciais'][i_cond]
            amplitude = solucao['amplitudes'][i_cond, i_sistema]
            energia_total = solucao['energia_mecanica'][-1, i_cond, i_sistema]
            
            rotulo = (f"<b>{sistemas_descricao[i_sistema]}</b><br>" +
                     f"f = {freq:.3f} Hz<br>" +
                     f"T = {1.0/freq:.3f} s<br>" +
                     f"x₀ = {x0:.3f} m, v₀ = {v0:.3f} m/s<br>" +
                     f"A = {amplitude:.3f} m<br>" +
                     f"E = {energia_total:.3f} J/kg")
            
            # cria nome legível para a legenda (mostra apenas o primeiro de cada sistema)
            if i_cond == 0:  # mostra apenas um item por sistema na legenda
                nome = f"Sistema {i_sistema}: ω={omega:.3f} rad/s"
                show_legend = True
            else:
                nome = f"Traj_S{i_sistema}_C{i_cond}"
                show_legend = False
            
            fig.add_trace(go.Scatter3d(
                x=solucao['posicao'][:, i_cond, i_sistema],
                y=solucao['velocidade'][:, i_cond, i_sistema],
                z=solucao['tempo'],
                mode='lines',
                line=dict(color=cor, width=1.5),
                name=nome,
                legendgroup=f'sistema_{i_sistema}',
                showlegend=show_legend,
                opacity=0.9,
                hovertemplate=rotulo + '<br>Posição: %{x:.3f} m<br>Velocidade: %{y:.3f} m/s<br>Tempo: %{z:.3f} s<extra></extra>'
            ))
    
    fig.update_layout(
        title=(
            "Espaço de Fases<br>" +
            f"<sup>{n_sistemas} sistemas | "
            f"{n_condicoes} condições iniciais por sistema | Total de {n_sistemas * n_condicoes} trajetórias</sup>"
        ),
        scene=dict(
            xaxis_title="Posição (m)",
            yaxis_title="Velocidade (m/s)",
            zaxis_title="Tempo (s)",
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=1.2),
                up=dict(x=0, y=0, z=1)
            )
        ),
        width=1200,
        height=900,
        legend=dict(
            title="Sistemas",
            yanchor="top",
            y=0.80,
            xanchor="left",
            x=0.80,
            bgcolor="rgba(255, 255, 255, 0.9)",
            bordercolor="Black",
            borderwidth=1,
            font=dict(size=12)
        ),
        hoverlabel=dict(
            bgcolor="white",
            font_size=16,
            font_family="Arial"
        )
    )
    
    return fig

def cria_grafico_2d(solucao, sistemas_descricao):
    """
    Cria um único gráfico 2D com todas as trajetórias de todos os sistemas no espaço de fases
    """
    fig = go.Figure()
    
    n_sistemas = solucao['n_sistemas']
    n_condicoes = solucao['n_condicoes']
    
    for i_sistema in range(n_sistemas):
        cor = CORES_PALETA[i_sistema % len(CORES_PALETA)]
        freq = solucao['frequencias_lineares'][i_sistema]
        omega = solucao['frequencias_angulares'][i_sistema]
        
        for i_cond in range(n_condicoes):
            x0, v0 = solucao['condicoes_iniciais'][i_cond]
            amplitude = solucao['amplitudes'][i_cond, i_sistema]
            
            # cria nome legível para a legenda (mostra apenas o primeiro de cada sistema)
            if i_cond == 0:  # mostra apenas um item por sistema na legenda
                nome = f"Sistema {i_sistema}: ω={omega:.3f} rad/s"
                show_legend = True
            else:
                nome = f"Traj_S{i_sistema}_C{i_cond}"
                show_legend = False
            
            fig.add_trace(go.Scatter(
                x=solucao['posicao'][:, i_cond, i_sistema],
                y=solucao['velocidade'][:, i_cond, i_sistema],
                mode='lines',
                line=dict(color=cor, width=1.0),
                name=nome,
                legendgroup=f'sistema_{i_sistema}',
                showlegend=show_legend,
                opacity=0.8,
                hovertemplate=(
                    f"<b>{sistemas_descricao[i_sistema]}</b><br>" +
                    f"f = {freq:.3f} Hz<br>" +
                    f"T = {1.0/freq:.3f} s<br>" +
                    f"x₀ = {x0:.3f} m, v₀ = {v0:.3f} m/s<br>" +
                    f"A = {amplitude:.3f} m<br>" +
                    f"Posição: %{{x:.3f}} m<br>" +
                    f"Velocidade: %{{y:.3f}} m/s<br>" +
                    f"<extra></extra>"
                )
            ))
    
    fig.update_layout(
        title=(
            "Espaço de Fases<br>" +
            f"<sup>{n_sistemas} sistemas | "
            f"{n_condicoes} condições iniciais por sistema | Total de {n_sistemas * n_condicoes} trajetórias</sup>"
        ),
        xaxis_title="Posição (m)",
        yaxis_title="Velocidade (m/s)",
        width=1400,
        height=1000,
        legend=dict(
            title="Sistemas",
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=1.00,
            bgcolor="rgba(255, 255, 255, 0.9)",
            bordercolor="Black",
            borderwidth=1,
            font=dict(size=12)
        ),
        hoverlabel=dict(
            bgcolor="white",
            font_size=16,
            font_family="Arial"
        ),
        plot_bgcolor='black',
        paper_bgcolor='black',
        xaxis=dict(
            showgrid=False,
            gridcolor='darkgray',
            zeroline=True,
            zerolinecolor='white',
            zerolinewidth=2,
            title_font_color='white',
            tickfont_color='white'
        ),
        yaxis=dict(
            showgrid=False,
            gridcolor='darkgray',
            zeroline=True,
            zerolinecolor='white',
            zerolinewidth=2,
            title_font_color='white',
            tickfont_color='white'
        ),
        title_font_color='white'
    )
    
    return fig

def main():
    """
    Função principal
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Dispositivo: {device}")
    
    # ============================================================
    # CONFIGURAÇÕES DA SIMULAÇÃO
    # ============================================================
    
    # parâmetros gerais
    n_sistemas = 10               # número de sistemas diferentes
    n_condicoes_por_sistema = 10  # número de condições iniciais por sistema
    num_periodos = 1              # número de períodos a serem simulados
    dt = 0.01                     # passo de tempo
    
    # intervalos físicos
    omega_min = 0.2              # frequência angular mínima (rad/s) - sistema lento
    omega_max = 15.0             # frequência angular máxima (rad/s) - sistema rápido
    x0_min, x0_max = -6.0, 6.0   # intervalo de posição inicial (m)
    v0_min, v0_max = -6.0, 6.0   # intervalo de velocidade inicial (m/s)
    
    # garante reprodutibilidade das simulações
    seed = 42
    
    print("\n=== MOSTRANDO CONFIGURAÇÕES DA SIMULAÇÃO ===")
    print(f"Número de sistemas: {n_sistemas}")
    print(f"Número de condições iniciais por sistema: {n_condicoes_por_sistema}")
    print(f"Total de simulações: {n_sistemas * n_condicoes_por_sistema}")
    print(f"Intervalo de frequência angular: [{omega_min}, {omega_max}] rad/s")
    print(f"Intervalo de posição inicial: [{x0_min}, {x0_max}] m")
    print(f"Intervalo de velocidade inicial: [{v0_min}, {v0_max}] m/s")
    print(f"Períodos simulados por sistema: {num_periodos}")
    
    # ============================================================
    # GERAÇÃO ALEATÓRIA DE CONFIGURAÇÕES
    # ============================================================
    
    print("\n=== GERANDO CONFIGURAÇÕES ALEATÓRIAS ===")
    
    condicoes_iniciais = gera_condicoes_iniciais_aleatorias(
        n_simulacoes=n_condicoes_por_sistema,
        x0_limites=(x0_min, x0_max),
        v0_limites=(v0_min, v0_max),
        seed=seed
    )

    frequencias_angulares = gera_frequencias_angulares_aleatorias(
        n_sistemas=n_sistemas,
        omega_min=omega_min,
        omega_max=omega_max,
        seed=seed
    )
    
    # cria descrições para cada sistema
    sistemas_descricao = []
    for i, omega in enumerate(frequencias_angulares):
        if omega < 1.0:
            tipo = "Lento"
        elif omega < 3.0:
            tipo = "Médio"
        elif omega < 8.0:
            tipo = "Rápido"
        else:
            tipo = "Muito Rápido"
        sistemas_descricao.append(tipo)
        
    print(f"\nCondições iniciais geradas:")
    x0_vals = condicoes_iniciais[:, 0].numpy()
    v0_vals = condicoes_iniciais[:, 1].numpy()
    print(f"  x₀: min={x0_vals.min():.3f}, max={x0_vals.max():.3f}, média={x0_vals.mean():.3f}")
    print(f"  v₀: min={v0_vals.min():.3f}, max={v0_vals.max():.3f}, média={v0_vals.mean():.3f}")
    
    print(f"\nFrequências angulares geradas:")
    print(f"  ω: min={min(frequencias_angulares):.3f}, max={max(frequencias_angulares):.3f}, méda={np.mean(frequencias_angulares):.3f}")

    # ============================================================
    # CRIA OSCILADOR E SIMULA TODOS OS SISTEMAS
    # ============================================================
    
    print("\n=== CRIANDO OSCILADOR PARA MÚLTIPLOS SISTEMAS ===")
    
    oscilador = OsciladorHarmonicoPyTorch(
        frequencias_angulares=frequencias_angulares,
        device=device
    )
    
    # encontra o índice do sistema com menor frequência angular (maior período)
    idx_sistema_lento = np.argmin(frequencias_angulares)
    periodo_lento = oscilador.periodos.cpu().numpy()[idx_sistema_lento]
    
    # calcula o tempo final baseado no período do sistema mais lento
    t_final_calculado = num_periodos * periodo_lento
    
    # ajusta t_final para garantir que o número de passos seja inteiro e feche o ciclo
    n_passos = int(np.ceil(t_final_calculado / dt))
    t_final = n_passos * dt
    
    print(f"Sistema mais lento (ID {idx_sistema_lento}): ω = {frequencias_angulares[idx_sistema_lento]:.3f} rad/s, T = {periodo_lento:.3f} s")
    print(f"t_final calculado: {t_final_calculado:.3f} s")
    print(f"t_final ajustado: {t_final:.3f} s")
    print(f"Número de passos: {n_passos}")
    print(f"dt = {dt} s")
    print(f"Sistema lento completará exatamente {num_periodos} período(s)")
    print(f"Período efetivo do sistema lento: {t_final/num_periodos:.3f} s")
    print(f"Erro relativo no período: {abs(t_final/num_periodos - periodo_lento)/periodo_lento*100:.6f}%")
    
    print(f"Simulando {num_periodos} período(s) completo(s): t_final = {t_final:.3f} s")
    print(f"Número de sistemas processados simultaneamente: {oscilador.n_sistemas}")
    print(f"Número de condições iniciais por sistema: {n_condicoes_por_sistema}")
    print(f"Total de trajetórias: {oscilador.n_sistemas * n_condicoes_por_sistema}")
    
    # ============================================================
    # GERANDO BASE DE DADOS
    # ============================================================
    
    print("\n=== GERANDO BASE DE DADOS ===")
    
    df, solucao = oscilador.gera_base_dados(
        condicoes_iniciais, t_final, dt, 
        sistemas_descricao
    )
    
    print(f"Registros gerados: {len(df):,}".replace(',', '.'))
    
    # ============================================================
    # SALVA BASE DE DADOS
    # ============================================================
    
    print("\n=== SALVANDO BASE DE DADOS ===")
    
    df.to_csv('base_ohs.csv', index=False, sep=';', encoding='utf-8')
    print(f"Total de registros na base: {len(df):,}".replace(',', '.'))
    print(f"Total de simulações: {n_sistemas * n_condicoes_por_sistema}")
    print(f"Sistemas incluídos: {df['sistema_id'].nunique()}")
    
    print("\n=== PRIMEIRAS 5 LINHAS DA BASE CONSOLIDADA ===")
    print(df.head(5))
    
    print("\n=== DISTRIBUIÇÃO DOS SISTEMAS NA BASE ===")
    distribuicao = df.groupby(['sistema_id', 'descricao_sistema']).size().reset_index(name='registros')
    for _, row in distribuicao.iterrows():
        print(f"  ID {int(row['sistema_id'])} - {row['descricao_sistema'][:50]}: {row['registros']:,} registros".replace(',', '.'))
    
    # ============================================================
    # CRIA VISUALIZAÇÕES
    # ============================================================
    
    print("\n=== CRIANDO VISUALIZAÇÕES ===")
    
    fig3d = cria_grafico_3d(solucao, sistemas_descricao)
    fig3d.show()
    
    fig2d = cria_grafico_2d(solucao, sistemas_descricao)
    fig2d.show()

    return df, solucao, sistemas_descricao

if __name__ == "__main__":
    df, solucao, sistemas_descricao = main()

Dispositivo: cpu

=== MOSTRANDO CONFIGURAÇÕES DA SIMULAÇÃO ===
Número de sistemas: 10
Número de condições iniciais por sistema: 10
Total de simulações: 100
Intervalo de frequência angular: [0.2, 15.0] rad/s
Intervalo de posição inicial: [-6.0, 6.0] m
Intervalo de velocidade inicial: [-6.0, 6.0] m/s
Períodos simulados por sistema: 1

=== GERANDO CONFIGURAÇÕES ALEATÓRIAS ===

Condições iniciais geradas:
  x₀: min=-5.551, max=5.318, média=-0.157
  v₀: min=-4.859, max=5.149, média=0.055

Frequências angulares geradas:
  ω: min=0.754, max=14.568, méda=7.630

=== CRIANDO OSCILADOR PARA MÚLTIPLOS SISTEMAS ===
Sistema mais lento (ID 2): ω = 0.754 rad/s, T = 8.330 s
t_final calculado: 8.330 s
t_final ajustado: 8.330 s
Número de passos: 833
dt = 0.01 s
Sistema lento completará exatamente 1 período(s)
Período efetivo do sistema lento: 8.330 s
Erro relativo no período: 0.004694%
Simulando 1 período(s) completo(s): t_final = 8.330 s
Número de sistemas processados simultaneamente: 10
Número de condi